# Adım 1 — Docker Ortamının KurulumuBu notebook, projenin altyapısını oluşturan **Docker Compose** mimarisini belgeler.## Mimari Özet```┌─────────────┐     ┌─────────────┐     ┌─────────────────────────┐│  Zookeeper  │────▶│    Kafka     │────▶│  Kafka Producer         ││  :2181      │     │  :9092/:29092│     │  (Python — CSV → JSON)  │└─────────────┘     └──────┬──────┘     └─────────────────────────┘                           │                    ┌──────▼──────┐     ┌─────────────┐                    │ Spark Master│────▶│ Spark Worker │                    │ :8080       │     │ (1 instans)  │                    └──────┬──────┘     └─────────────┘                           │              ┌────────────┼────────────┐              ▼            ▼            ▼        ┌──────────┐ ┌──────────┐ ┌──────────┐        │  MLflow  │ │ Jupyter  │ │Dashboard │        │  :5000   │ │  :8888   │ │:8501/:5001│        └──────────┘ └──────────┘ └──────────┘```

In [ ]:
# docker-compose.yml dosyasını okuyup servisleri listeleimport yaml, oscompose_path = os.path.join(os.pardir, "docker-compose.yml")with open(compose_path) as f:    compose = yaml.safe_load(f)print(f"Toplam servis sayısı: {len(compose.get('services', {}))}")print(f"\n{'Servis':<25} {'Image / Build':<40} {'Portlar'}")print("─" * 90)for name, cfg in compose.get("services", {}).items():    image = cfg.get("image", "—")    if image == "—" and "build" in cfg:        b = cfg["build"]        image = b.get("dockerfile", str(b)) if isinstance(b, dict) else str(b)    ports = ", ".join(cfg.get("ports", [])) or "—"    print(f"{name:<25} {str(image):<40} {ports}")

## Servisler ve Görevleri| Servis | Teknoloji | Görevi ||--------|-----------|-------|| **zookeeper** | Bitnami Zookeeper | Kafka broker koordinasyonu || **kafka** | Bitnami Kafka | Mesaj kuyruğu — `iot-network-traffic` topic || **kafka-producer** | Python 3.9 + kafka-python | CSV → JSON mesaj üretici (ayarlanabilir hız) || **spark-master** | Bitnami Spark 3.x | Spark cluster yöneticisi || **spark-worker** | Bitnami Spark 3.x | İş yürütücü (executor) || **mlflow-server** | MLflow 2.x | Deney takibi, model kaydı || **jupyter** | JupyterLab | Notebook ortamı (EDA, sunum) || **dashboard-api** | Flask + kafka-python | SSE canlı veri API'si || **streamlit-dashboard** | Streamlit + Plotly | İnteraktif dashboard |## Dockerfile Özeti| Dockerfile | Base Image | Ek Kurulumlar ||------------|-----------|---------------|| `Dockerfile.spark` | bitnami/spark | Delta Lake JAR'ları, PySpark, MLflow, kafka-python || `Dockerfile.producer` | python:3.9-slim | kafka-python || `Dockerfile.dashboard` | python:3.9-slim | Flask, flask-cors, deltalake, pyarrow |## Ağ ve Volume Yapısı- **Network:** `bigdata-net` (bridge) — tüm servisler aynı ağda- **Volumes:**  - `delta-storage/` → Bronze / Silver / Gold Delta tabloları  - `mlruns/` → MLflow experiment veritabanı  - `data/raw/` → Edge-IIoTset CSV dosyası

In [ ]:
# Dockerfile içeriklerini gösterfor dockerfile in ["docker/Dockerfile.spark", "docker/Dockerfile.producer", "docker/Dockerfile.dashboard"]:    path = os.path.join(os.pardir, dockerfile)    if os.path.exists(path):        print(f"\n{'═'*60}")        print(f"  {dockerfile}")        print(f"{'═'*60}")        with open(path) as f:            print(f.read())

## Çalıştırma```bash# Tüm stack'i başlatdocker compose up -d# Kontroldocker compose ps# Logları izledocker compose logs -f spark-master```| Servis | Erişim Adresi ||--------|--------------|| JupyterLab | http://localhost:8888 || Spark Master UI | http://localhost:8080 || MLflow UI | http://localhost:5000 || Dashboard | http://localhost:8501 || Dashboard API | http://localhost:5001 || Kafka (dış) | localhost:29092 |